In [ ]:
#训练
import numpy as np
from Utils.data import *
from Utils.train import *
from Utils.metric import *
from Model.model import *
from sklearn import preprocessing

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data as Data
from torch.utils.data import DataLoader, TensorDataset
import gc

def main_rnn():
    print("hi")
    fn=["Cluster3"]
    for name in range (len(fn)):
        filename=fn[name]
        data=loadtxtmethod(f"./Data/{filename}.dat")
        subdata=data.loc[:,['Z', 'E0', 'kt', 'AST', 'Gcsky', 'delta_ktc', 'kde', 'daily_kt', 'kcsi', 'psi', 'hourly_kt', 'kds60', 'aod.clim', 'alb.clim', 'cld.clim','kd', 'kb']]
        subdata=subdata.astype('float32')
        subdata=np.array(subdata)
        del data
        gc.collect()
        subdata=subdata[:,[0,1,3,4,5,6,7,8,9,10,11,12,13,14,2,15,16]]
    
        min_max_scaler = preprocessing.MinMaxScaler()
        norm_data1 = min_max_scaler.fit_transform(subdata[:,0:14])
        norm_data=np.concatenate((norm_data1,subdata[:,14:17]),axis=1)
        
        min_=min_max_scaler.data_min_
        max_=min_max_scaler.data_max_
        num_train=int(norm_data.shape[0]*0.6)
        num_val=int(norm_data.shape[0]*0.2)
       
        train_dataset=norm_data[0:num_train,:]
        val_dataset=norm_data[num_train:num_train+num_val,:]
        test_dataset=norm_data[num_train+num_val:,:]
        test_obser=subdata[num_train+num_val:,(subdata.shape[1]-2):subdata.shape[1]]
        
        seed_torch(1)
        Hidden=[10]
        all_val_loss=[]
        for param1 in range (len(Hidden)):
            print(f"***********************File{filename} Hid{Hidden[param1]}************************")
            shuffle_train_data =True
            lr=0.001
            batch_size=10000
            epochs=1500
            hid=Hidden[param1]
            cnn_dim1=4
            cnn_dim2=4
            k1=2
            k2=2
            rat=1
            ffn_hidden=hid
            n_head=1
            n_layers=1
            device = torch.device('cuda:0')
            criterion = proposed_loss().to(device)
            model = Proposed_model(15,hid,cnn_dim1,cnn_dim2,k1,k2,rat,device,7,ffn_hidden,n_head,n_layers)
            model = model.to(device)
            optimizer = optim.Adam(model.parameters(),lr)
            val_loss=train_proposed(epochs, model, train_dataset,val_dataset,optimizer,criterion,device,batch_size,hid,filename) 
            all_val_loss.append(val_loss)
            torch.save(model.state_dict(), f'Model_saving/Proposed_{filename}_hidden{hid}_ep1000.pt')
            pred_result=predict_proposed(model,device,test_dataset)
            pred_result=proposed_convert(pred_result,norm_data[num_train+num_val:,14])
            kd_metric=point_forecast(pred_result[:,0],test_obser[:,0])
            kb_metric=point_forecast(pred_result[:,1],test_obser[:,1])
            type_index=np.repeat(np.array(["kd","kb"]),pred_result.shape[0])
            result=np.concatenate((np.transpose(pred_result).flatten()[:,np.newaxis],np.transpose(test_obser).flatten()[:,np.newaxis],type_index[:,np.newaxis]),axis=1)
            dataf = pd.DataFrame(result,columns=["Pred","Ob","Type_index"])
            dataf.to_csv(f"Result/Proposed_{filename}_hidden{hid}.csv")
            torch.cuda.empty_cache()
        del subdata   
if __name__=="__main__":
    main_rnn() 